# <span style="color:#ff8c00">05 · VLM zero-shot baselines</span>

Two open vision-language models — Gemma 4 E4B (bf16) and Qwen3.5-9B (FP8) — answer the `val2014` questions directly in vLLM, normalised and scored with the same official VQA metric as the classifiers.

Modes (`MODE` below, in run order): `smoke` (50 questions) · `pilot` (2,000 questions) · `full` (all 214,354 questions) · `report` (official accuracy tables).

In [ ]:
MODE = "smoke"        # smoke (50 questions) | pilot (2,000) | full (all 214,354) | report
MODELS = ["gemma4_e4b", "qwen35_9b_fp8"]   # keys of vqa_lab.vlm.VLM_MODELS, answered one at a time
LIMIT = {"smoke": 50, "pilot": 2000, "full": None}   # questions per mode (sampled with seed 123)
BATCH = 512           # questions handed to vLLM per call (it batches them on the GPU itself)
PREFIX_CACHING = True # reuse an image's tokens across its questions (off once in the pilot, to compare)

# <span style="color:#ff8c00">Setup</span>

Paths and the val2014 questions with their ten human answers.

In [ ]:
import subprocess

from vqa_lab.config import load_paths
from vqa_lab.data import fetch_hub, read_table
from vqa_lab.report import write_report
from vqa_lab.vlm import VLM_MODELS, answer_command, question_file, vlm_tables

paths = load_paths()
val = read_table(paths, "val")
print(f"{len(val):,} val questions; mode {MODE}; models {MODELS}")

# <span style="color:#ff8c00">Weights</span>

Downloaded once into `data/raw_files/models/<org>/<name>`.

In [ ]:
for key in MODELS:
    repo = VLM_MODELS[key]["weights"]
    fetch_hub(repo, paths.raw_dir / "models" / repo)

# <span style="color:#ff8c00">1 · Answers</span>

One model at a time on the GPU. Answers are saved in chunks of 20,000 questions, so a stopped run resumes where it was.

In [ ]:
if MODE != "report":
    questions = question_file(val, paths, limit=LIMIT[MODE])
    for key in MODELS:
        subprocess.run(answer_command(paths, key, questions, batch=BATCH, prefix_caching=PREFIX_CACHING), check=True)

# <span style="color:#ff8c00">2 · Report</span>

Every finished question set scored with the official metric, next to the best classifiers.

The report also merges every finished benchmark into one file, `benchmark_results.md`, next to the notebooks.

In [ ]:
runs, comparison = vlm_tables(paths)
display(comparison)
display(runs)
print("\nAll benchmarks merged into:", write_report(paths))